# 01 — Title/abstract screening

Screen a project's title/abstract queue. Every keystroke appends to
`decisions.jsonl` immediately, so closing this notebook — or losing the kernel —
costs at most the record on screen.

Records are presented in an order seeded from the project slug and uncorrelated
with citation count or any other ranking. That is deliberate: a ranked queue
imports the ranker's judgement into a protocol that is supposed to be yours.

Author names and citation counts are hidden by default for the same reason. Pass
`blind=False` if your protocol genuinely needs them.

In [ ]:
import os

import prismabib
from prismabib.project import Project

print(f"prismabib {prismabib.__version__}")

# Set PRISMABIB_NOTEBOOK_SLUG to screen your own project. The default is the
# bundled reference fixture, so this notebook executes in CI without a Scopus
# key and without touching anyone's real review.
SLUG = os.environ.get("PRISMABIB_NOTEBOOK_SLUG", "reference")
REVIEWER = os.environ.get("PRISMABIB_NOTEBOOK_REVIEWER", "reviewer")

project = Project.open(SLUG)
print(f"screening {project.slug} at {project.root}")

## Build the store if it is not there yet

Layer 1 is derived, never authored: it is rebuilt from the immutable Layer 0
archive by one function call, so deleting it costs nothing but time.

In [ ]:
from prismabib.store.load import build_store

if not project.db_path.exists():
    stats = build_store(project, rebuild=True)
    print(f"built: {stats.records_loaded} records")

## Where you are

Run this before screening and again whenever you want to see progress. It reads
the decision log, so it stays accurate across restarts.

In [ ]:
from prismabib.screening.queue import screening_queue
from prismabib.stage import PrismaStage

queue = screening_queue(project, PrismaStage.TITLE_ABSTRACT, REVIEWER)
print(f"{queue.decided} decided / {queue.total} total — {queue.remaining} remaining")

## Screen

`i` include · `e` then a digit exclude · `u` unsure · `n`/`p` move · `z` undo · `?` help

`unsure` never resolves to inclusion — the record stays in the queue for a
second pass and is reported separately in the PRISMA diagram, rather than being
folded into your exclusions.

In [ ]:
from prismabib.screening.ui import screener

screener(project, stage="title_abstract", reviewer=REVIEWER)

## Then

`prismabib flow <slug>` prints the PRISMA counts, drawn from the decision log —
no number in it is typed by hand.